# Step 7 — Export des qualifizierten Katalogs

Vereinigt den aktuellen Step-5-Output (Tier 1: aktuelle Nachtzug-Halte) mit den manuellen Step-6-Ergänzungen und schreibt eine CSV, die exakt dem Spaltenvertrag von `seed.py`s `_ONTD_SEED_CSV_COLUMNS` entspricht. Schlüssel ist die OSM-ID, sodass ein Stop, der auf beiden Wegen qualifiziert, nur einmal geschrieben wird und ein erneuter Step-5-Lauf direkt in den Katalog einfließt, ohne die manuelle Auswahl anzutasten.

Aus welcher Ebene ein Stop stammt, wird separat in `stop_seed_provenance.csv` geschrieben: Der Katalog selbst muss exakt dem Spaltenvertrag von `seed.py` entsprechen, daher kann die Provenienz nicht darin mitfahren. Ohne dieses Sidecar würde nichts auf der Platte zwischen einem Stop, den ein Nachtzug heute bedient, und einem manuell hinzugefügten unterscheiden.

**country** — stammt aus dem ONTD-Export via Step-4-Join, sofern vorhanden, da ONTD kuratierte nationale Daten sind; sonst der Wert der jeweiligen Quellebene. Steps 5 und 6 wenden dieselbe Präferenz bereits an, dies ist also ein Sicherheitsnetz und nicht die Stelle, an der die Korrektur passiert — Step 6 berichtet, was er geändert hat.

**stop_timezone** — aus dem Land als IANA-Name abgeleitet. Die alte Step-6-Datei trug stattdessen einen bloßen UTC-Offset, der keine Sommerzeit abbilden kann und für Irland falsch war (als +1 markiert); Schema und restlicher Katalog nutzen IANA-Namen.

**stop_charge_eur** stammt aus `charges/data/station_charges.csv`, generiert von den Kalibrierungs-Notebooks in `charges/`. Ein Stop, der dort fehlt, wird leer geschrieben, und `seed.py` setzt NULL, was über den Land-/Global-Default (aktuell 11.28 EUR) aufgelöst wird. Leer ist Absicht: eine Platzhalterzahl würde diesen Fallback überschreiben und die Frage "welche Stops brauchen noch echte Entgeltdaten?" unbeantwortbar machen, während NULL die Frage zu einer Einzeiler-Query macht.

## Imports

In [34]:
from __future__ import annotations

import csv
from pathlib import Path

from data_sources import DATA_DIR, ensure_local, local_input
from shapely.geometry import Point

import geopandas as gpd
import pandas as pd
import transliterate
import regex as re
from unidecode import unidecode
import gettext
import pycountry

import deepl
from functools import lru_cache

import re
import requests
from functools import lru_cache


## Pfade und Spaltenschema

In [2]:
OUTPUT_PATH = DATA_DIR / "stop_seed_catalog.csv"
PROVENANCE_PATH = DATA_DIR / "step6_manual_additions.csv"
COUNTRIES_SHP = "countrycode_data/ne_10m_admin_0_countries.shp"
auth_key = "bd943d88-12ef-436d-ab3b-bac8e2715132:fx"  # aus deinem DeepL-Account

#für countrycode matching
NEAREST_MAX_DISTANCE_M = 2000

# Generiert von charges/02_station_charges.ipynb, das die registrierten
# Tarifdokumente einliest. Gitignored wie jedes andere Kalibrierungs-Artefakt
# im Projekt — die Notebooks sind die Wahrheit, dies ist ihr Output.
CHARGES_PATH = (
    Path.cwd() / "charges" / "data" / "station_charges.csv"
)

# seed.py::_ONTD_SEED_CSV_COLUMNS — im Gleichschritt halten.
SEED_COLUMNS = [
    "stop_id",
    "stop_name",
    "country_code",
    "stop_timezone",
    "stop_lat",
    "stop_lon",
    "stop_charge_eur",
]

In [3]:
#step6 laden und relevante Spalten wählen & zu Geodataframe umwandeln
step6_output = pd.read_csv(PROVENANCE_PATH, encoding="utf-8-sig")
stopdata = step6_output[[
    "stop_id",
    "stop_name",
    "stop_timezone",
    "stop_lat",
    "stop_lon"]]

geometry = [Point(xy) for xy in zip(stopdata["stop_lon"], stopdata["stop_lat"])]
stops_gdf = gpd.GeoDataFrame(stopdata.copy(), geometry=geometry, crs="EPSG:4326")

#shapefile mit ländergrenzen ladenb
countries = gpd.read_file(COUNTRIES_SHP).to_crs("EPSG:4326")

## Countrycode hinzufügen

In [4]:
def resolve_iso_a3(row):
    for field in ("ISO_A3", "ISO_A3_EH"):
        value = row.get(field)
        if value and value != "-99":
            return value
    return None

countries["country_code"] = countries.apply(resolve_iso_a3, axis=1)

MANUAL_ISO_FIXES = {
    "Norway": "NOR", "Kosovo": "XKX",
    "N. Cyprus": "CYP",
}
missing_iso = countries["country_code"].isna()
countries.loc[missing_iso, "country_code"] = countries.loc[missing_iso, "ADMIN"].map(MANUAL_ISO_FIXES)

countries_slim = countries[["country_code", "ADMIN", "geometry"]].rename(columns={"ADMIN": "country_name"})

In [5]:
missing_iso = countries["country_code"].isna()

In [6]:
#in welchen land leigen punkte + countrycode hinzufügen
joined = gpd.sjoin(stops_gdf, countries_slim, how="left", predicate="within")
joined = joined.drop(columns=["index_right"], errors="ignore")

In [7]:
#nähestes Land wählen für die die nciht gematcht werden konnten
unmatched_mask = joined["country_code"].isna()
unmatched = joined[unmatched_mask].copy()

if len(unmatched):
    unmatched_proj = unmatched.drop(columns=["country_code", "country_name"]).to_crs("EPSG:3857")
    countries_proj = countries_slim.to_crs("EPSG:3857")

    nearest = gpd.sjoin_nearest(
        unmatched_proj, countries_proj, how="left",
        max_distance=NEAREST_MAX_DISTANCE_M, distance_col="distance_m",
    )
    nearest = nearest.drop(columns=["index_right"], errors="ignore")

    for col in ["country_code", "country_name"]:
        joined.loc[unmatched_mask, col] = nearest[col].values

still_unmatched = joined[joined["country_code"].isna()]
if len(still_unmatched):
    ohne_land = joined[joined["country_code"].isna()]
    print(f"{len(ohne_land)} Stops ohne Land:")
    print(ohne_land[["stop_id", "stop_name", "stop_lat", "stop_lon"]])

stopdata_with_country = joined.drop(columns="geometry").reset_index(drop=True)

3 Stops ohne Land:
             stop_id        stop_name   stop_lat   stop_lon
583  osm:n5526331885  Bodø - Bådåddjo  67.286319  14.390816
585  osm:n5526332038           Narvik  68.441550  17.441113
591  osm:n5720557872        Stavanger  58.966577   5.732216


In [8]:
#manuell countrycode hinzufügen -> stopID von oben rauskopieren


MANUAL_COUNTRY_FIXES = {
    # "stop_id": "XX",
    "osm:n5526331885" : "NOR",
    "osm:n5526332038" : "NOR",
    "osm:n5720557872" : "NOR"
}
mask = stopdata_with_country["stop_id"].isin(MANUAL_COUNTRY_FIXES)
stopdata_with_country.loc[mask, "country_code"] = stopdata_with_country.loc[mask, "stop_id"].map(MANUAL_COUNTRY_FIXES)

In [9]:
stopdata_with_country.head()

,stop_id,stop_name,stop_timezone,stop_lat,stop_lon,country_code,country_name
0,osm:n25546152,Pécs,1,46.066366,18.225331,HUN,Hungary
1,osm:n13895194676,Terminali i Transportit Publik Tiranë,1,41.347259,19.777025,ALB,Albania
2,osm:n13895194677,Durrës,1,41.317741,19.456562,ALB,Albania
3,osm:n1613271652,Prishtinë,1,42.658934,21.151067,XKX,Kosovo
4,osm:n2107256271,Ferizaj,1,42.368744,21.153630,XKX,Kosovo


## ganzer Ländername + Übersetzungen

In [10]:

LANGUAGES = {
    "country_en": None,   # Englisch = Originalname, keine Übersetzung nötig
    "country_de": "de",
    "country_fr": "fr",
    "country_nl": "nl",
    "country_it": "it",
    "country_es": "es",
    "country_pl": "pl",
}

MANUAL_OVERRIDES = {
    "XKX": {
        "country_en": "Kosovo",
        "country_de": "Kosovo",
        "country_fr": "Kosovo",
        "country_nl": "Kosovo",
        "country_it": "Kosovo",
        "country_es": "Kosovo",
        "country_pl": "Kosowo",
    },
    "MDA": {  # Moldova
        "country_en": "Moldova",
        "country_de": "Moldau",
        "country_fr": "Moldavie",
        "country_nl": "Moldavië",
        "country_it": "Moldavia",
        "country_es": "Moldavia",
        "country_pl": "Mołdawia",
    },
    "TUR": {  # Türkei
        "country_en": "Turkey",
        "country_de": "Türkei",
        "country_fr": "Turquie",
        "country_nl": "Turkije",
        "country_it": "Turchia",
        "country_es": "Turquía",
        "country_pl": "Turcja",
    },
}

# Übersetzungsobjekte einmal vorab laden (nicht pro Zeile neu)
_translations = {}
for lang in set(LANGUAGES.values()):
    if lang is None:
        continue
    _translations[lang] = gettext.translation(
        "iso3166-1", pycountry.LOCALES_DIR, languages=[lang]
    )

def get_country_names(country_code):
    if not isinstance(country_code, str):
        return {col: None for col in LANGUAGES}

    if country_code in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[country_code]

    country = pycountry.countries.get(alpha_3=country_code)
    if country is None:
        return {col: None for col in LANGUAGES}

    result = {}
    for col, lang in LANGUAGES.items():
        if lang is None:
            result[col] = country.name
        else:
            result[col] = _translations[lang].gettext(country.name)
    return result

# Anwendung auf den DataFrame
names_df = stopdata_with_country["country_code"].apply(get_country_names).apply(pd.Series)
stopdata_with_country = pd.concat([stopdata_with_country, names_df], axis=1)


In [11]:
# Duplikate an Spaltennamen erkennen
print(stopdata_with_country.columns[stopdata_with_country.columns.duplicated()].tolist())

# Nur die erste Instanz jeder Spalte behalten, Rest löschen
stopdata_with_country = stopdata_with_country.loc[:, ~stopdata_with_country.columns.duplicated()]

[]


In [12]:
stopdata_with_country.filter(regex="^country").isnull().sum()

country_code    0
country_name    3
country_en      0
country_de      0
country_fr      0
country_nl      0
country_it      0
country_es      0
country_pl      0
dtype: int64

In [ ]:
stopdata_with_country = stopdata_with_country.drop(columns=["country_name"])

## Länder-Zeitzonen

Eine IANA-Zone pro Land. Aus dem Land abgeleitet statt aus einem UTC-Offset, damit die Sommerzeit von der Zonendatenbank gehandhabt wird, statt in den Daten eingefroren zu sein.

In [13]:
COUNTRY_TIMEZONES = {
    "ALB": "Europe/Tirane",
    "AUT": "Europe/Vienna",
    "BIH": "Europe/Sarajevo",
    "BEL": "Europe/Brussels",
    "BGR": "Europe/Sofia",
    "BLR": "Europe/Minsk",
    "CHE": "Europe/Zurich",
    "CZE": "Europe/Prague",
    "DEU": "Europe/Berlin",
    "DNK": "Europe/Copenhagen",
    "EST": "Europe/Tallinn",
    "ESP": "Europe/Madrid",
    "FIN": "Europe/Helsinki",
    "FRA": "Europe/Paris",
    "GBR": "Europe/London",
    "GRC": "Europe/Athens",
    "HRV": "Europe/Zagreb",
    "HUN": "Europe/Budapest",
    "IRL": "Europe/Dublin",
    "ITA": "Europe/Rome",
    "LTU": "Europe/Vilnius",
    "LUX": "Europe/Luxembourg",
    "LVA": "Europe/Riga",
    "MDA": "Europe/Chisinau",
    "MNE": "Europe/Podgorica",
    "MKD": "Europe/Skopje",
    "NLD": "Europe/Amsterdam",
    "NOR": "Europe/Oslo",
    "POL": "Europe/Warsaw",
    "PRT": "Europe/Lisbon",
    "ROU": "Europe/Bucharest",
    "SRB": "Europe/Belgrade",
    "RUS": "Europe/Moscow",
    "SWE": "Europe/Stockholm",
    "SVN": "Europe/Ljubljana",
    "SVK": "Europe/Bratislava",
    "TUR": "Europe/Istanbul",
    "UKR": "Europe/Kyiv",
    "XKX": "Europe/Belgrade",
}

stopdata_with_country["stop_timezone"] = stopdata_with_country["country_code"].map(COUNTRY_TIMEZONES)

## Ausgeschlossene Stop-IDs

OSM-IDs, deren Step-4-Match Ausschuss ist — die OSM-Station ist ein unbenanntes oder einbuchstabiges Objekt Hunderte Kilometer vom ONTD-Stop entfernt, dem sie zugeordnet wurde. Ausgeschlossen statt mit falschem Standort geseedet.

In [14]:
EXCLUDED_STOP_IDS = {
    "osm:n4896717721",  # ONTD Dağkadı Hızlı Tren İstasyonu -> OSM "tren", 2743 km
    "osm:n8515238217",  # ONTD Tekučica -> OSM "A", 355 km
    "osm:n9553124517",  # ONTD Közép-Garadna -> OSM "Arad", 222 km
}

## Stopnamen lateinisch (latin_name) & unicode (ascii_name)

In [15]:
def transliterate_name(name_latin, country_code):
    # Fallback für lateinische Transliteration (Kyrillisch)
    if bool(re.search(r'\p{Cyrillic}', name_latin)):
        if country_code == "SRB":  # Serbien
            name_latin = transliterate.translit(name_latin, 'sr', reversed=True)
        elif country_code == "BGR":  # Bulgarien
            name_latin = transliterate.translit(name_latin, 'bg', reversed=True)
        elif country_code == "MKD":  # Nordmazedonien
            name_latin = transliterate.translit(name_latin, 'mk', reversed=True)
        elif country_code == "UKR":  # Ukraine
            name_latin = transliterate.translit(name_latin, 'uk', reversed=True)
        elif country_code == "MDA":  # Moldawien (Transnistrien)
            name_latin = transliterate.translit(name_latin, 'ru', reversed=True)
        elif country_code == "BIH":  # Bosnien und Herzegowina (Republika Srpska)
            name_latin = transliterate.translit(name_latin, 'sr', reversed=True)

    # Fallback für griechische Transliteration
    if bool(re.search(r'\p{Greek}', name_latin)):
        name_latin = transliterate.translit(name_latin, 'el', reversed=True)

    return name_latin

In [16]:
#lateinisch
stopdata_with_country["latin_name"] = stopdata_with_country.apply(
    lambda row: transliterate_name(row["stop_name"], row["country_code"]),
    axis=1
)

In [17]:
# unicode (ascii)
stopdata_with_country["ascii_name"] = stopdata_with_country["latin_name"].apply(unidecode)

In [18]:
stopdata_with_country["ascii_name"].isnull().sum()

np.int64(0)

## Stopnamen übersetzen mit deepl (Api hat einmaliges Limit von 1.000.000 Zeichen)

In [65]:

translator = deepl.Translator(auth_key)

TARGET_LANGS = {
    "stopname_en": "EN-GB",
    "stopname_de": "DE",
    "stopname_fr": "FR",
    "stopname_nl": "NL",
    "stopname_it": "IT",
    "stopname_es": "ES",
    "stopname_pl": "PL",
}

@lru_cache(maxsize=None)
def translate_cached(name, lang):
    if not isinstance(name, str) or not name.strip():
        return name
    try:
        result = translator.translate_text(name, target_lang=lang, source_lang=None)
        return result.text
    except Exception:
        return name 

In [44]:


for col, lang in TARGET_LANGS.items():
    print(f"\n--- Übersetze nach {lang} ({col}) ---")
    start = time.time()

    results = []
    total = len(stopdata_with_country)
    for i, name in enumerate(stopdata_with_country["latin_name"]):
        results.append(translate_cached(name, lang))
        if i % 50 == 0 or i == total - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            remaining = (total - i - 1) / rate if rate > 0 else 0
            print(f"  {i+1}/{total} ({elapsed:.0f}s vergangen, ~{remaining:.0f}s verbleibend)")

    stopdata_with_country[col] = results
    print(f"{col}: fertig nach {time.time() - start:.1f} Sekunden")


--- Übersetze nach EN-GB (stopname_en) ---
  1/944 (0s vergangen, ~389s verbleibend)
  51/944 (6s vergangen, ~113s verbleibend)
  101/944 (44s vergangen, ~365s verbleibend)
  151/944 (69s vergangen, ~363s verbleibend)
  201/944 (98s vergangen, ~362s verbleibend)
  251/944 (126s vergangen, ~348s verbleibend)
  301/944 (154s vergangen, ~329s verbleibend)
  351/944 (184s vergangen, ~311s verbleibend)
  401/944 (212s vergangen, ~287s verbleibend)
  451/944 (241s vergangen, ~264s verbleibend)
  501/944 (270s vergangen, ~239s verbleibend)
  551/944 (298s vergangen, ~212s verbleibend)
  601/944 (326s vergangen, ~186s verbleibend)
  651/944 (356s vergangen, ~160s verbleibend)
  701/944 (384s vergangen, ~133s verbleibend)
  751/944 (410s vergangen, ~105s verbleibend)
  801/944 (438s vergangen, ~78s verbleibend)
  851/944 (465s vergangen, ~51s verbleibend)
  901/944 (494s vergangen, ~24s verbleibend)
  944/944 (517s vergangen, ~0s verbleibend)
stopname_en: fertig nach 516.5 Sekunden


In [46]:
#manuelle Korrektur der Stopnamen die wörtlich übersetzt wurden
STOPNAME_OVERRIDES = {'osm:n1065130141': {'stopname_de': 'Kónya', 'stopname_es': 'Kónya', 'stopname_fr': 'Kónya', 'stopname_it': 'Kónya', 'stopname_nl': 'Kónya', 'stopname_pl': 'Kónya'},
 'osm:n10783341033': {'stopname_de': 'Gaia', 'stopname_es': 'Gaia', 'stopname_fr': 'Gaia', 'stopname_it': 'Gaia', 'stopname_nl': 'Gaia', 'stopname_pl': 'Gaia'},
 'osm:n10880998201': {'stopname_de': 'Rijeka', 'stopname_es': 'Rijeka', 'stopname_fr': 'Rijeka', 'stopname_it': 'Rijeka', 'stopname_nl': 'Rijeka', 'stopname_pl': 'Rijeka'},
 'osm:n10935384072': {'stopname_de': 'Valence-Ville', 'stopname_es': 'Valence-Ville', 'stopname_it': 'Valence-Ville', 'stopname_nl': 'Valence-Ville', 'stopname_pl': 'Valence-Ville'},
 'osm:n10940370468': {'stopname_de': 'Cerbère', 'stopname_es': 'Cerbère', 'stopname_it': 'Cerbère', 'stopname_nl': 'Cerbère', 'stopname_pl': 'Cerbère'},
 'osm:n10940403868': {'stopname_de': 'Gap', 'stopname_es': 'Gap', 'stopname_fr': 'Gap', 'stopname_it': 'Gap', 'stopname_nl': 'Gap', 'stopname_pl': 'Gap'},
 'osm:n12048036719': {'stopname_es': 'Orte', 'stopname_fr': 'Orte', 'stopname_it': 'Orte', 'stopname_nl': 'Orte', 'stopname_pl': 'Orte'},
 'osm:n1216887890': {'stopname_de': 'Rivne', 'stopname_es': 'Rivne', 'stopname_fr': 'Rivne', 'stopname_it': 'Rivne', 'stopname_nl': 'Rivne', 'stopname_pl': 'Rivne'},
 'osm:n12214361791': {'stopname_de': 'Varna', 'stopname_es': 'Varna', 'stopname_fr': 'Varna', 'stopname_it': 'Varna', 'stopname_nl': 'Varna', 'stopname_pl': 'Varna'},
 'osm:n12299676794': {'stopname_de': 'Savona', 'stopname_es': 'Savona', 'stopname_fr': 'Savona', 'stopname_it': 'Savona', 'stopname_nl': 'Savona', 'stopname_pl': 'Savona'},
 'osm:n12966105377': {'stopname_de': 'Iskar', 'stopname_es': 'Iskar', 'stopname_fr': 'Iskar', 'stopname_it': 'Iskar', 'stopname_nl': 'Iskar', 'stopname_pl': 'Iskar'},
 'osm:n13715292382': {'stopname_de': 'Duved', 'stopname_es': 'Duved', 'stopname_fr': 'Duved', 'stopname_it': 'Duved', 'stopname_nl': 'Duved', 'stopname_pl': 'Duved'},
 'osm:n1601596124': {'stopname_de': 'Niš', 'stopname_es': 'Niš', 'stopname_fr': 'Niš', 'stopname_it': 'Niš', 'stopname_nl': 'Niš', 'stopname_pl': 'Niš'},
 'osm:n166747734': {'stopname_es': 'Kolari'},
 'osm:n1687607528': {'stopname_de': 'Vác', 'stopname_es': 'Vác', 'stopname_fr': 'Vác', 'stopname_it': 'Vác', 'stopname_nl': 'Vác'},
 'osm:n1696546782': {'stopname_es': 'Septemvri', 'stopname_fr': 'Septemvri', 'stopname_it': 'Septemvri', 'stopname_nl': 'Septemvri', 'stopname_pl': 'Septemvri'},
 'osm:n194740791': {'stopname_de': 'Pamiers', 'stopname_es': 'Pamiers', 'stopname_it': 'Pamiers', 'stopname_nl': 'Pamiers', 'stopname_pl': 'Pamiers'},
 'osm:n1974456578': {'stopname_es': 'Łódź Kaliska', 'stopname_fr': 'Łódź Kaliska'},
 'osm:n2321100569': {'stopname_de': 'Pirdop', 'stopname_es': 'Pirdop', 'stopname_fr': 'Pirdop', 'stopname_it': 'Pirdop', 'stopname_nl': 'Pirdop', 'stopname_pl': 'Pirdop'},
 'osm:n256277342': {'stopname_de': 'Sărățel', 'stopname_es': 'Sărățel', 'stopname_nl': 'Sărățel'},
 'osm:n2726063373': {'stopname_de': 'Adana', 'stopname_es': 'Adana', 'stopname_fr': 'Adana', 'stopname_it': 'Adana', 'stopname_nl': 'Adana', 'stopname_pl': 'Adana'},
 'osm:n2770844472': {'stopname_de': 'Hamar', 'stopname_es': 'Hamar', 'stopname_fr': 'Hamar', 'stopname_it': 'Hamar', 'stopname_nl': 'Hamar', 'stopname_pl': 'Hamar'},
 'osm:n2824346770': {'stopname_de': 'Chop', 'stopname_es': 'Chop', 'stopname_fr': 'Chop', 'stopname_it': 'Chop', 'stopname_nl': 'Chop', 'stopname_pl': 'Chop'},
 'osm:n287554537': {'stopname_de': 'Larisa', 'stopname_es': 'Larisa', 'stopname_fr': 'Larisa', 'stopname_it': 'Larisa', 'stopname_nl': 'Larisa', 'stopname_pl': 'Larisa'},
 'osm:n289897536': {'stopname_de': 'Ungheni', 'stopname_es': 'Ungheni', 'stopname_fr': 'Ungheni', 'stopname_it': 'Ungheni', 'stopname_nl': 'Ungheni', 'stopname_pl': 'Ungheni'},
 'osm:n2966093688': {'stopname_de': 'Novi Sad', 'stopname_es': 'Novi Sad', 'stopname_fr': 'Novi Sad', 'stopname_it': 'Novi Sad', 'stopname_nl': 'Novi Sad', 'stopname_pl': 'Novi Sad'},
 'osm:n29830753': {'stopname_de': 'Konin', 'stopname_es': 'Konin', 'stopname_fr': 'Konin', 'stopname_it': 'Konin', 'stopname_pl': 'Konin'},
 'osm:n3080695733': {'stopname_es': 'Baden', 'stopname_fr': 'Baden', 'stopname_pl': 'Baden'},
 'osm:n308841776': {'stopname_de': 'Xanthi', 'stopname_es': 'Xanthi', 'stopname_fr': 'Xanthi', 'stopname_it': 'Xanthi', 'stopname_nl': 'Xanthi', 'stopname_pl': 'Xanthi'},
 'osm:n3459469757': {'stopname_de': 'Piła Główna', 'stopname_nl': 'Piła Główna'},
 'osm:n3466186493': {'stopname_de': 'Dax', 'stopname_es': 'Dax', 'stopname_fr': 'Dax', 'stopname_it': 'Dax', 'stopname_nl': 'Dax', 'stopname_pl': 'Dax'},
 'osm:n3916678066': {'stopname_de': 'Hendaye', 'stopname_es': 'Hendaye', 'stopname_fr': 'Hendaye', 'stopname_it': 'Hendaye', 'stopname_nl': 'Hendaye', 'stopname_pl': 'Hendaye'},
 'osm:n394710073': {'stopname_es': 'Vannes'},
 'osm:n4290857026': {'stopname_de': 'Metz', 'stopname_fr': 'Metz', 'stopname_it': 'Metz', 'stopname_nl': 'Metz', 'stopname_pl': 'Metz'},
 'osm:n43174364': {'stopname_de': 'Breda'},
 'osm:n46756926': {'stopname_de': 'Faro', 'stopname_es': 'Faro', 'stopname_fr': 'Faro', 'stopname_it': 'Faro', 'stopname_nl': 'Faro', 'stopname_pl': 'Faro'},
 'osm:n4883927548': {'stopname_de': 'Serrai', 'stopname_es': 'Serrai', 'stopname_fr': 'Serrai', 'stopname_it': 'Serrai', 'stopname_nl': 'Serrai', 'stopname_pl': 'Serrai'},
 'osm:n529102526': {'stopname_es': 'Oświęcim'},
 'osm:n5598384401': {'stopname_de': 'Caen', 'stopname_it': 'Caen', 'stopname_pl': 'Caen'},
 'osm:n564634911': {'stopname_de': 'Hel', 'stopname_fr': 'Hel', 'stopname_it': 'Hel', 'stopname_pl': 'Hel'},
 'osm:n5648124921': {'stopname_de': 'Most', 'stopname_es': 'Most', 'stopname_fr': 'Most', 'stopname_it': 'Most', 'stopname_nl': 'Most', 'stopname_pl': 'Most'},
 'osm:n5668860678': {'stopname_de': 'Pula', 'stopname_es': 'Pula', 'stopname_fr': 'Pula', 'stopname_it': 'Pula', 'stopname_nl': 'Pula', 'stopname_pl': 'Pula'},
 'osm:n612493471': {'stopname_de': 'Turku', 'stopname_es': 'Turku', 'stopname_fr': 'Turku', 'stopname_it': 'Turku', 'stopname_nl': 'Turku'},
 'osm:n6175162599': {'stopname_de': 'Katerini', 'stopname_es': 'Katerini', 'stopname_fr': 'Katerini', 'stopname_it': 'Katerini', 'stopname_nl': 'Katerini', 'stopname_pl': 'Katerini'},
 'osm:n6528546131': {'stopname_de': 'Sumy', 'stopname_es': 'Sumy', 'stopname_fr': 'Sumy', 'stopname_it': 'Sumy', 'stopname_nl': 'Sumy'},
 'osm:n6605149663': {'stopname_de': 'Reading', 'stopname_es': 'Reading', 'stopname_fr': 'Reading', 'stopname_it': 'Reading', 'stopname_nl': 'Reading', 'stopname_pl': 'Reading'},
 'osm:n7066101885': {'stopname_nl': 'Nitra'},
 'osm:n7198645523': {'stopname_de': 'Bar', 'stopname_es': 'Bar', 'stopname_nl': 'Bar', 'stopname_pl': 'Bar'},
 'osm:n727366743': {'stopname_de': 'Krzyż', 'stopname_es': 'Krzyż', 'stopname_fr': 'Krzyż', 'stopname_it': 'Krzyż', 'stopname_nl': 'Krzyż'},
 'osm:n7472007553': {'stopname_de': 'Levanto', 'stopname_fr': 'Levanto', 'stopname_it': 'Levanto', 'stopname_nl': 'Levanto', 'stopname_pl': 'Levanto'},
 'osm:n7512362062': {'stopname_de': 'Figueres', 'stopname_es': 'Figueres', 'stopname_fr': 'Figueres', 'stopname_it': 'Figueres', 'stopname_nl': 'Figueres', 'stopname_pl': 'Figueres'},
 'osm:n7606786768': {'stopname_de': 'Assen', 'stopname_es': 'Assen'},
 'osm:n783848373': {'stopname_de': "Uman'", 'stopname_fr': "Uman'", 'stopname_nl': "Uman'", 'stopname_pl': "Uman'"},
 'osm:n8072437221': {'stopname_de': 'Basmane', 'stopname_es': 'Basmane', 'stopname_fr': 'Basmane', 'stopname_it': 'Basmane', 'stopname_nl': 'Basmane', 'stopname_pl': 'Basmane'},
 'osm:n8220188327': {'stopname_es': 'Rosynka', 'stopname_fr': 'Rosynka', 'stopname_it': 'Rosynka', 'stopname_nl': 'Rosynka'},
 'osm:n842367835': {'stopname_de': 'Potenza Centrale', 'stopname_es': 'Potenza Centrale', 'stopname_nl': 'Potenza Centrale', 'stopname_pl': 'Potenza Centrale'},
 'osm:n8459380277': {'stopname_de': 'Yambol', 'stopname_es': 'Yambol', 'stopname_fr': 'Yambol', 'stopname_it': 'Yambol', 'stopname_nl': 'Yambol', 'stopname_pl': 'Yambol'},
 'osm:n8519451211': {'stopname_de': 'Pau', 'stopname_es': 'Pau', 'stopname_fr': 'Pau', 'stopname_it': 'Pau', 'stopname_nl': 'Pau', 'stopname_pl': 'Pau'},
 'osm:n8594635347': {'stopname_de': 'Vinnytsia'},
 'osm:n8745537418': {'stopname_de': 'Lille-Europe', 'stopname_es': 'Lille-Europe', 'stopname_it': 'Lille-Europe', 'stopname_nl': 'Lille-Europe', 'stopname_pl': 'Lille-Europe'},
 'osm:n8745537419': {'stopname_de': 'Lille-Flandres', 'stopname_es': 'Lille-Flandres', 'stopname_fr': 'Lille-Flandres', 'stopname_it': 'Lille-Flandres', 'stopname_nl': 'Lille-Flandres', 'stopname_pl': 'Lille-Flandres'},
 'osm:n8930884280': {'stopname_es': 'Boden C', 'stopname_fr': 'Boden C', 'stopname_it': 'Boden C', 'stopname_nl': 'Boden C', 'stopname_pl': 'Boden C'},
 'osm:n9212525414': {'stopname_de': 'Dubno', 'stopname_es': 'Dubno', 'stopname_it': 'Dubno'},
 'osm:n9330871706': {'stopname_de': 'Deva', 'stopname_es': 'Deva', 'stopname_fr': 'Deva', 'stopname_it': 'Deva', 'stopname_nl': 'Deva', 'stopname_pl': 'Deva'},
 'osm:n9553124517': {'stopname_de': 'Arad', 'stopname_es': 'Arad', 'stopname_fr': 'Arad', 'stopname_it': 'Arad', 'stopname_nl': 'Arad', 'stopname_pl': 'Arad'},
 'osm:n9555790979': {'stopname_fr': 'Satu Mare'},
 'osm:n9643537166': {'stopname_de': 'Volos', 'stopname_es': 'Volos', 'stopname_fr': 'Volos', 'stopname_it': 'Volos', 'stopname_nl': 'Volos', 'stopname_pl': 'Volos'},
 'osm:n9698173348': {'stopname_de': 'Jasinja', 'stopname_es': 'Jasinja', 'stopname_fr': 'Jasinja', 'stopname_it': 'Jasinja', 'stopname_nl': 'Jasinja', 'stopname_pl': 'Jasinja'},
 'osm:w272187224': {'stopname_de': 'Dej Triaj', 'stopname_es': 'Dej Triaj', 'stopname_fr': 'Dej Triaj', 'stopname_it': 'Dej Triaj', 'stopname_nl': 'Dej Triaj', 'stopname_pl': 'Dej Triaj'},
 'osm:w86123754': {'stopname_de': 'Lugo', 'stopname_es': 'Lugo', 'stopname_fr': 'Lugo', 'stopname_it': 'Lugo', 'stopname_nl': 'Lugo', 'stopname_pl': 'Lugo'}}

def apply_stopname_overrides(df, overrides=STOPNAME_OVERRIDES, id_col="stop_id"):
    """Wendet manuelle Korrekturen auf fehlerhafte DeepL-Uebersetzungen von Bahnhofsnamen an."""
    for stop_id, col_values in overrides.items():
        mask = df[id_col] == stop_id
        for col, value in col_values.items():
            df.loc[mask, col] = value
    return df

stopdata_with_country = apply_stopname_overrides(stopdata_with_country)

def apply_stopname_overrides(df, overrides=STOPNAME_OVERRIDES, id_col="stop_id"):
    for stop_id, col_values in overrides.items():
        mask = df[id_col] == stop_id
        for col, value in col_values.items():
            df.loc[mask, col] = value
    return df

stopdata_with_country = apply_stopname_overrides(stopdata_with_country)

In [52]:
#Hauptbahnhof & Hbf vereinheitlichen
stopdata_with_country["stopname_de"] = stopdata_with_country["stopname_de"].str.replace(
    r"\bHbf\b", "Hauptbahnhof", regex=True
)

## Städte mit Übersetzungen hinzufügen

In [69]:

# ISO-3166-Alpha3 -> DeepL-Quellsprachcode (statt Autodetect)
COUNTRY_SOURCE_LANG = {
    "AUT": "DE", "DEU": "DE", "CHE": "DE", "LUX": "FR",
    "CZE": "CS", "POL": "PL", "HUN": "HU", "SVK": "SK",
    "FRA": "FR", "BEL": "NL", "NLD": "NL",
    "ITA": "IT", "ESP": "ES", "PRT": "PT",
    "GRC": "EL", "BGR": "BG", "ROU": "RO", "MDA": "RO",
    "SRB": None, "HRV": None,
    "SVN": "SL", "UKR": "UK", "RUS": "RU",
    "SWE": "SV", "NOR": None, "FIN": "FI", "DNK": "DA",
    "LTU": "LT", "LVA": "LV", "EST": "ET",
    "GBR": "EN", "IRL": "EN",
    "TUR": "TR",
}

CITY_SUFFIX_PATTERNS = [
    r"\bhlavní nádraží\b", r"\bHauptbahnhof\b", r"\bHbf\b",
    r"\bGlavni kolodvor\b", r"\bGłówny\b", r"\bGłówna\b",
    r"\bCentrálā? stacija\b", r"\bcentralstation\b", r"\bCentraal\b",
    r"\bCentrale\b", r"\bcentral\b", r"\bCentrum\b", r"\bcentar\b",
    r"\bGara de Nord\b", r"\bvokzal\b",
]

def extract_city(name):
    for pattern in CITY_SUFFIX_PATTERNS:
        match = re.search(pattern, name, flags=re.IGNORECASE)
        if match:
            city = name[:match.start()].strip(" -–,")
            if city:
                return city
    return name

city_base = stopdata_with_country["latin_name"].apply(extract_city)

# translate_cached mit source_lang-Parameter (3 Argumente)
from functools import lru_cache

@lru_cache(maxsize=None)
def translate_cached_src(name, lang, source_lang):
    if not isinstance(name, str) or not name.strip():
        return name
    try:
        result = translator.translate_text(name, target_lang=lang, source_lang=source_lang)
        return result.text
    except Exception:
        return name

CITY_LANG_COLS = {
    "city_en": "EN-GB",
    "city_de": "DE",
    "city_fr": "FR",
    "city_nl": "NL",
    "city_it": "IT",
    "city_es": "ES",
    "city_pl": "PL",
}

for col, lang in CITY_LANG_COLS.items():
    print(f"\n--- Stadtnamen nach {lang} ({col}) ---")
    start = time.time()
    results = []
    total = len(stopdata_with_country)
    for i, (city, country_code) in enumerate(zip(city_base, stopdata_with_country["country_code"])):
        source_lang = COUNTRY_SOURCE_LANG.get(country_code)
        results.append(translate_cached_src(city, lang, source_lang))
        if i % 50 == 0 or i == total - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            remaining = (total - i - 1) / rate if rate > 0 else 0
            print(f"  {i+1}/{total} ({elapsed:.0f}s vergangen, ~{remaining:.0f}s verbleibend)")
    stopdata_with_country[col] = results
    print(f"{col}: fertig nach {time.time() - start:.1f} Sekunden")


--- Stadtnamen nach EN-GB (city_en) ---
  1/944 (0s vergangen, ~448s verbleibend)
  51/944 (9s vergangen, ~164s verbleibend)
  101/944 (45s vergangen, ~379s verbleibend)
  151/944 (71s vergangen, ~373s verbleibend)
  201/944 (99s vergangen, ~366s verbleibend)
  251/944 (127s vergangen, ~350s verbleibend)
  301/944 (155s vergangen, ~330s verbleibend)
  351/944 (182s vergangen, ~307s verbleibend)
  401/944 (243s vergangen, ~329s verbleibend)
  451/944 (272s vergangen, ~297s verbleibend)
  501/944 (300s vergangen, ~265s verbleibend)
  551/944 (327s vergangen, ~233s verbleibend)
  601/944 (349s vergangen, ~199s verbleibend)
  651/944 (375s vergangen, ~169s verbleibend)
  701/944 (403s vergangen, ~140s verbleibend)
  751/944 (429s vergangen, ~110s verbleibend)
  801/944 (472s vergangen, ~84s verbleibend)
  851/944 (503s vergangen, ~55s verbleibend)
  901/944 (532s vergangen, ~25s verbleibend)
  944/944 (556s vergangen, ~0s verbleibend)
city_en: fertig nach 556.1 Sekunden

--- Stadtnamen na

In [90]:
STATION_WORDS = {
    "city_en": [
        "main station", "central station", "railway station", "railway",
        "station", "north station", "south station", "east station",
        "west station", "terminal", "public transport", "north", "south",
        "east", "west", "town", "city", "travel centre", "travel center",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_de": [
        "Hauptbahnhof", "Hbf", "Zentralbahnhof", "Bahnhof", "Ostbahnhof",
        "Westbahnhof", "Südbahnhof", "Nordbahnhof",
        "Öffentlicher Verkehrsknotenpunkt", "Verkehrsknotenpunkt",
        "Süd", "Nord", "Ost", "West", "Stadt", "Reisezentrum",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_fr": [
        "gare routière", "gare centrale", "gare du nord", "gare du sud",
        "gare de l'est", "gare de l'ouest", "gare", "routière",
        "Sud", "Nord", "Est", "Ouest", "Ville",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_nl": [
        "Openbaarvervoersterminal", "Centraalstation", "centraal station", "centraal", "station",
        "Zuid", "Noord", "Oost", "West", "Stad", "Reiscentrum",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_it": [
        "Terminal dei trasporti pubblici", "stazione centrale",
        "stazione nord", "stazione sud", "stazione",
        "Sud", "Nord", "Est", "Ovest", "Città", "Centro viaggi",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_es": [
        "Terminal de transporte público", "estación central",
        "estación del norte", "estación del sur", "estación",
        "Sur", "Norte", "Este", "Oeste", "Ciudad",
        "estación de tren de", "autobuses de",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
    "city_pl": [
        "Terminal transportu publicznego", "dworzec główny",
        "dworzec centralny", "dworzec kolejowy", "dworzec", "stacja",
        "kolejowy", "Główny", "Główna", "Miasto", "Centrum turystyczne",
        "Północny", "Północna", "Północne", "Północ",
        "Południowy", "Południowa", "Południowe", "Południe",
        "Wschodni", "Wschodnia", "Wschodnie", "Wschód",
        "Zachodni", "Zachodnia", "Zachodnie", "Zachód",
        "Stathmos", "Vokzal", "Vokzal'na", "Gara", "Kolodvor",
    ],
}

# Praepositionen/Artikel, die nach dem Entfernen eines Wortes uebrig bleiben koennen
LEADING_TRAILING_WORDS = ["de", "di", "du", "des", "w", "von", "van", "d'"]

def strip_station_words(name, words):
    if not isinstance(name, str):
        return name
    cleaned = name
    for word in sorted(words, key=len, reverse=True):  # laengere Begriffe zuerst
        cleaned = re.sub(rf"\b{re.escape(word)}\b", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" -–,'")

    # uebrig gebliebene Praeposition am Anfang/Ende entfernen (z.B. "de Mersin" -> "Mersin")
    tokens = cleaned.split()
    while tokens and tokens[0].lower().strip("'") in LEADING_TRAILING_WORDS:
        tokens.pop(0)
    while tokens and tokens[-1].lower().strip("'") in LEADING_TRAILING_WORDS:
        tokens.pop()
    cleaned = " ".join(tokens)

    return cleaned if cleaned else name  # falls alles entfernt wurde: Original behalten

for col, words in STATION_WORDS.items():
    stopdata_with_country[col] = stopdata_with_country[col].apply(
        lambda x: strip_station_words(x, words)
    )

In [92]:
stopdata_with_country.to_csv(
    r"C:\Users\glanz\Documents\BackOnTrack\night-train-target-network\backend\models\infrastructure\stops\data\step6_translated.csv",
    index=False
)

## ... weiter noch nicht bearbeitet (Johanna) ...

## Funktion: Stationsentgelte laden 

mit KI csv Datei erstellen (mit OSM_id und stop_cost) & Pfad hier angeben
-> stop_cost: netto, stop_tax: Steuersatz, stop_cost_incl_tax: brutto

In [ ]:
#Länder bei denen keine Stopksoten anfallen: BEL, CZE, DNK, EST, FRA, GRC, HRV, IRL, LUX, LVA, NOR, POL, SWE
#Länder fehlen noch: HUN, ALB, XKX, TUR, CHE, BIH, MNE, SRB, SVN, MKD, BGR, ROU, ITA, NLD, ESP, FIN, UKR, MDA, LTU, GBR, RUS
# Pfad zur Kosten-CSV je Land (ISO-3166-Alpha3 -> Dateipfad)
COUNTRY_COST_FILES = {
    #in Google Drive hochgeladen
    "DEU": r"",
    "AUT": r"",
    "PRT": r"",
    "SVK": r""
}

# Manuell gepflegter Steuersatz je Land, wenn nichts eingetragen -> Steuersatz = 0%
STOP_TAX = {
    "DEU": 0.19,
    "AUT": 0.20,
    "PRT": 0.23,
    "SVK": 0.23

}
def load_country_costs(country_cost_files):
    stop_id_to_cost = {}
    for country_code, filepath in country_cost_files.items():
        try:
            df = pd.read_csv(filepath, dtype={"osm_id": str})
        except FileNotFoundError:
            print(f"Warnung: Datei fuer {country_code} nicht gefunden: {filepath}")
            continue
        for _, row in df.iterrows():
            stop_id_to_cost[str(row["osm_id"])] = row["cost"]
    return stop_id_to_cost

def add_stop_cost_and_tax(df, country_cost_files=COUNTRY_COST_FILES,
                           id_col="stop_id", country_col="country_code"):
    stop_id_to_cost = load_country_costs(country_cost_files)

    df["stop_cost"] = df[id_col].astype(str).map(stop_id_to_cost)
    df["stop_tax"] = df[country_col].map(STOP_TAX).fillna(0)
    return df

stopdata_with_country = add_stop_cost_and_tax(stopdata_with_country)

stopdata_with_country["stop_cost_incl_tax"] = (
    stopdata_with_country["stop_cost"] * (1 + stopdata_with_country["stop_tax"])
)

## Funktion: Float parsen

In [ ]:
def parse_float(value):
    if value is None:
        return None
    text = str(value).strip().replace(",", ".")
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None

## Funktion: ONTD-Länder laden

In [ ]:
def load_ontd_countries() -> dict[str, str]:
    """OSM-Stop-ID -> ONTD-Ländercode, aus dem Step-4-Join."""
    path = ensure_local("step4_MatchingONTDtoOSM.csv")
    countries = {}
    with open(path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            osm_id = (row.get("osm_stop_id") or "").strip()
            country = (row.get("ontd_country") or "").strip().upper()
            if osm_id and country:
                countries.setdefault(osm_id, country)
    return countries

## Funktion: Kandidaten iterieren (Step 5, dann Step 6)

In [ ]:
def iter_candidates():
    """(source, stop_id, stop_name, source_country, lat, lon, reason) aus
    Step 5, dann Step 6. Step 5 zuerst, damit seine ONTD-gestützte Zeile
    gewinnt, wenn ein Stop auf beiden Wegen qualifiziert."""
    step5_path = local_input(
        "step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"
    )
    with open(step5_path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            yield (
                "step5",
                (row.get("osm_stop_id") or "").strip(),
                (row.get("osm_stop_name") or row.get("ontd_name") or "").strip(),
                (row.get("ontd_country") or "").strip().upper(),
                parse_float(row.get("osm_lat")),
                parse_float(row.get("osm_lon")),
                f"night_train_stop:{(row.get('schedule_name') or '').strip()}",
            )
    step6_path = local_input(
        "step6_manual_additions.csv", "step6_manual_additions.ipynb"
    )
    with open(step6_path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            yield (
                "step6_manual",
                (row.get("stop_id") or "").strip(),
                (row.get("stop_name") or "").strip(),
                (row.get("country") or "").strip().upper(),
                parse_float(row.get("stop_lat")),
                parse_float(row.get("stop_lon")),
                (row.get("reason") or "").strip(),
            )

## Daten laden: ONTD-Länder und Entgelte

In [ ]:
ontd_countries = load_ontd_countries()
charges = load_station_charges()

## Kandidaten verarbeiten: Katalog- und Provenienz-Zeilen bauen

Dedupliziert per OSM-ID, filtert ausgeschlossene Stops, löst Land/Zeitzone auf und sammelt Stops ohne auflösbare Zeitzone für den Abbruch am Ende.

In [ ]:
rows, unknown_tz = [], []
skipped: dict[str, str] = {}
seen_ids = set()
per_source = {"step5": 0, "step6_manual": 0}
provenance = []

for (
    source,
    stop_id,
    stop_name,
    source_country,
    lat,
    lon,
    reason,
) in iter_candidates():
    if not stop_id or stop_id in seen_ids:
        continue
    if stop_id in EXCLUDED_STOP_IDS:
        skipped.setdefault(stop_id, stop_name)
        continue
    seen_ids.add(stop_id)
    per_source[source] += 1

    country = ontd_countries.get(stop_id, source_country)

    timezone = COUNTRY_TIMEZONES.get(country)
    if timezone is None:
        # Kein Land bedeutet keine Zeitzone, und ein ohne sie geschriebener
        # Stop wäre falsch statt nur unvollständig — daher hier gesammelt und
        # unten ausgelöst, damit er nie unbemerkt aus dem Katalog verschwindet.
        unknown_tz.append((stop_id, stop_name, country))
        continue

    if lat is None or lon is None:
        skipped[stop_id] = f"{stop_name} (fehlende Koordinaten)"
        continue

    provenance.append(
        {
            "stop_id": stop_id,
            "stop_name": stop_name,
            "source": source,
            "reason": reason,
        }
    )
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": stop_name,
            "country_code": country,
            "stop_timezone": timezone,
            "stop_lat": f"{lat:.7f}",
            "stop_lon": f"{lon:.7f}",
            "stop_charge_eur": (
                "" if stop_id not in charges else f"{charges[stop_id]:.2f}"
            ),
        }
    )

## Katalog und Provenienz schreiben

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=SEED_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

with open(PROVENANCE_PATH, "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(
        fh, fieldnames=["stop_id", "stop_name", "source", "reason"]
    )
    writer.writeheader()
    writer.writerows(provenance)

print(
    f"{len(rows)} Stops nach {OUTPUT_PATH.name} geschrieben "
    f"({per_source['step5']} aus Step 5, "
    f"{per_source['step6_manual']} manuelle Step-6-Ergänzungen)"
)

## Zusammenfassung und Warnungen

Bricht am Ende mit `SystemExit` ab, falls Stops ohne auflösbare Zeitzone übrig blieben — damit ein solcher Stop nie unbemerkt aus dem Katalog verschwindet.

In [ ]:
unexplained = sum(
    1 for p in provenance if p["source"] == "step6_manual" and not p["reason"]
)
if unexplained:
    print(
        f"  {unexplained} manuelle Ergänzungen tragen keinen Grund "
        f"— siehe step6_manual_additions.ipynb"
    )
if skipped:
    print(f"übersprungen {len(skipped)}: {skipped}")
unknown_charges = sorted(set(charges) - seen_ids)
if unknown_charges:
    print(
        f"  WARNUNG: {len(unknown_charges)} Entgelt-Zeile(n) nennen einen Stop, "
        f"der nicht im Katalog ist — veraltete IDs in {CHARGES_PATH.name}: "
        f"{unknown_charges[:5]}"
    )

if unknown_tz:
    countries = sorted({country for _, _, country in unknown_tz if country})
    blank = [(i, n) for i, n, country in unknown_tz if not country]
    raise SystemExit(
        f"{len(unknown_tz)} Stop(s) mangels Zeitzone verworfen.\n"
        + (
            f"  keine Zuordnung für {countries} — zu COUNTRY_TIMEZONES hinzufügen\n"
            if countries
            else ""
        )
        + (
            f"  gar kein Land: {blank} — upstream in Step 6 beheben\n"
            if blank
            else ""
        )
    )